In [ ]:
from datasets import load_dataset

ds_corpus = load_dataset("vidore/vidore_v3_industrial", "corpus")
ds_labels = load_dataset("vidore/vidore_v3_industrial", "qrels")
ds_queries = load_dataset("vidore/vidore_v3_industrial", "queries")

In [ ]:
# Keep only the English queries.
ds_queries["test"] = ds_queries["test"].filter(lambda lang: lang == "english", input_columns="language")

kept_ids = set(ds_queries["test"]["query_id"])
ds_labels["test"] = ds_labels["test"].filter(lambda qid: qid in kept_ids, input_columns="query_id")
print(f"kept {len(ds_queries['test'])} English queries and {len(ds_labels['test'])} qrels rows")

In [ ]:
!pip install -q bm25s PyStemmer sentence-transformers transformers

## First-stage retrieval

The reranker needs candidates, so this reuses the hybrid retriever from `rag_text.ipynb` (BM25 + Qwen3-Embedding fused with RRF) over each page's `markdown`. The page embeddings are loaded from the `.npy` cache that notebook wrote.

In [ ]:
import numpy as np
import torch
from collections import defaultdict
from pathlib import Path
from tqdm.auto import tqdm
import bm25s
import Stemmer
from sentence_transformers import SentenceTransformer

# Read columns directly: iterating rows would decode every page image just to get the id and text.
page_ids = ds_corpus["test"]["corpus_id"]
page_texts = [text or "" for text in ds_corpus["test"]["markdown"]]
page_index = {pid: i for i, pid in enumerate(page_ids)}  # corpus_id -> position in page_texts

qrels_by_query = defaultdict(set)  # query_id -> relevant corpus_ids
for row in ds_labels["test"]:
    if row["score"] > 0:
        qrels_by_query[row["query_id"]].add(row["corpus_id"])


def to_pages(page_scores: np.ndarray, k: int) -> list[int]:
    return [page_ids[i] for i in np.argsort(-page_scores)[:k]]


# Sparse: BM25 over page markdown
stemmer = Stemmer.Stemmer("english")
bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize(page_texts, stopwords="en", stemmer=stemmer, show_progress=False), show_progress=False)


def sparse_pages(query: str, k: int) -> list[int]:
    q_tokens = bm25s.tokenize(query, stopwords="en", stemmer=stemmer, return_ids=False, show_progress=False)[0]
    return to_pages(bm25.get_scores(q_tokens), k)


# Dense: Qwen3-Embedding-0.6B, page embeddings cached by rag_text.ipynb
device = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_MODEL, MAX_SEQ_LENGTH = "Qwen/Qwen3-Embedding-0.6B", 2048
QUERY_PROMPT = "Instruct: Given a question about a technical manual, retrieve the page that answers it\nQuery:"
EMBED_CACHE = Path(f"page_emb_{EMBED_MODEL.split('/')[-1]}_{MAX_SEQ_LENGTH}.npy")

embedder = SentenceTransformer(EMBED_MODEL, device=device, tokenizer_kwargs={"padding_side": "left"})
embedder.max_seq_length = MAX_SEQ_LENGTH
if device == "cuda":
    embedder.half()

if EMBED_CACHE.exists() and np.load(EMBED_CACHE, mmap_mode="r").shape[0] == len(page_texts):
    page_emb = np.load(EMBED_CACHE)
else:
    page_emb = embedder.encode(page_texts, batch_size=4, normalize_embeddings=True, show_progress_bar=True)
    np.save(EMBED_CACHE, page_emb)


def dense_pages(query: str, k: int) -> list[int]:
    q_emb = embedder.encode(query, prompt=QUERY_PROMPT, normalize_embeddings=True)
    return to_pages(np.dot(page_emb, q_emb), k)  # unit vectors, so dot product = cosine similarity


# Hybrid: Reciprocal Rank Fusion of the two rankings
RRF_POOL, RRF_C = 50, 60


def hybrid_pages(query: str, k: int, pool: int = RRF_POOL, c: int = RRF_C) -> list[int]:
    fused = defaultdict(float)
    for ranked in (sparse_pages(query, pool), dense_pages(query, pool)):
        for rank, pid in enumerate(ranked, 1):
            fused[pid] += 1 / (c + rank)
    return sorted(fused, key=fused.get, reverse=True)[:k]

## Reranking (Qwen3-Reranker)

A cross-encoder style reranker: [`Qwen/Qwen3-Reranker-0.6B`](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B) reads the query and one page together and scores how likely the page answers the query (the probability of "yes" versus "no"). It is far more accurate than comparing separate embeddings but costs one forward pass per (query, page) pair, so it only reorders the top `RERANK_POOL` pages from the hybrid retriever.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

RERANK_MODEL = "Qwen/Qwen3-Reranker-0.6B"
RERANK_POOL = 20       # hybrid candidates passed to the reranker per query
RERANK_MAX_LEN = 1024  # prompt tokens per (query, page) pair; longer pages are truncated
RERANK_BATCH = 4
INSTRUCTION = "Given a question about a technical manual, judge whether the page answers it"

rr_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL, padding_side="left")
rr_model = AutoModelForCausalLM.from_pretrained(
    RERANK_MODEL, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device).eval()

# The reranker is a causal LM prompted to answer "yes" or "no"; the score is P("yes") at the last position.
YES_ID, NO_ID = rr_tokenizer.convert_tokens_to_ids("yes"), rr_tokenizer.convert_tokens_to_ids("no")
prefix_ids = rr_tokenizer.encode(
    '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct '
    'provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n',
    add_special_tokens=False,
)
suffix_ids = rr_tokenizer.encode("<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n", add_special_tokens=False)


@torch.no_grad()
def rerank_scores(query: str, texts: list[str]) -> list[float]:
    """P("yes") that each page text answers the query."""
    body_len = RERANK_MAX_LEN - len(prefix_ids) - len(suffix_ids)
    scores = []
    for i in range(0, len(texts), RERANK_BATCH):
        # Truncating the body cuts the page (last in the prompt), never the query or the prompt wrapper.
        bodies = rr_tokenizer(
            [f"<Instruct>: {INSTRUCTION}\n<Query>: {query}\n<Document>: {t}" for t in texts[i:i + RERANK_BATCH]],
            add_special_tokens=False, truncation=True, max_length=body_len,
        )["input_ids"]
        batch = rr_tokenizer.pad({"input_ids": [prefix_ids + b + suffix_ids for b in bodies]}, return_tensors="pt").to(device)
        logits = rr_model(**batch).logits[:, -1, [NO_ID, YES_ID]]
        scores += torch.log_softmax(logits.float(), dim=-1)[:, 1].exp().tolist()
    return scores


def rerank_pages(query: str, candidates: list[int], k: int | None = None) -> list[int]:
    """Reorder candidate corpus_ids by reranker score, best first."""
    scores = rerank_scores(query, [page_texts[page_index[pid]] for pid in candidates])
    return [candidates[i] for i in np.argsort(-np.array(scores))[:k]]

## Benchmark

Compares the hybrid top 5 with the top 5 after reranking the hybrid top `RERANK_POOL` pages, using page-level recall@5 and nDCG@5 against `qrels`. The pool recall printed below is the ceiling: reranking can only reorder pages the hybrid retriever already found.

In [ ]:
import math
import pandas as pd

K = 5
N_QUERIES = None  # set e.g. 50 for a quick run: reranking on CPU is slow


def metrics(ranked: list[int], rel: set[int], k: int = K) -> dict:
    dcg = sum(1 / math.log2(r + 1) for r, pid in enumerate(ranked[:k], 1) if pid in rel)
    idcg = sum(1 / math.log2(r + 1) for r in range(1, min(len(rel), k) + 1))
    return {f"recall@{k}": len(rel & set(ranked[:k])) / len(rel), f"nDCG@{k}": dcg / idcg}


rows = {"hybrid (RRF)": [], f"hybrid + rerank (top {RERANK_POOL})": []}
pool_recall = []
queries = [q for q in ds_queries["test"] if qrels_by_query.get(q["query_id"])][:N_QUERIES]

for q in tqdm(queries, desc="queries"):
    rel = qrels_by_query[q["query_id"]]
    candidates = hybrid_pages(q["query"], RERANK_POOL)
    pool_recall.append(len(rel & set(candidates)) / len(rel))
    rows["hybrid (RRF)"].append(metrics(candidates, rel))
    rows[f"hybrid + rerank (top {RERANK_POOL})"].append(metrics(rerank_pages(q["query"], candidates), rel))

results = pd.DataFrame({name: pd.DataFrame(r).mean() for name, r in rows.items()}).T
print(f"{len(queries)} queries, candidate pool recall@{RERANK_POOL} (reranking ceiling): {np.mean(pool_recall):.3f}")
results.round(3)